# Pre-processing a .CHA file for use in CE analysis

In [1]:
import os
import re
import sys
import pandas as pd
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
data_path = '../data'
chas_path = os.path.join(data_path, 'callfriend-ko/transcr')

tsvs_path = os.path.join(chas_path, 'tsvs')
if not os.path.exists(tsvs_path):
    os.mkdir(tsvs_path)

outputs_path = os.path.join(data_path, 'server_ready', 'callfriend-ko.parquet')

In [3]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

# Processing all CHA files

Note: the package used here was developed by Prof. Garber. Cite via:

Garber, L. (2019). CHA file python parser. Zenodo. https://doi.org/10.5281/zenodo.3364020

In [4]:
all_files = grab_all_files(chas_path, '.txt')
len(all_files), all_files[0]

(100, '../data/callfriend-ko/transcr/ko_4012.txt')

In [6]:
for f in tqdm(all_files):
    df = []
    
    new_file = f.split('/')[-1].replace('.txt', '.tsv')
    
    conversation_id = f.split('/')[-1].replace('.txt', '').split('_')[-1]
    corpus = f.split('/')[2]
    
    f_ = open(f, 'r', encoding='EUC-KR')
    TEXT = f_.read().split('\n\n')[1:]
    f_.close()
    for l in TEXT:
        if l:
            if '[[skip]]' not in l:
                speaker_break = list(re.finditer(r'\.\d+ [A-Z]+[0-9]*:', l))
                speaker_break = speaker_break[0].span()[-1]
                df += [
                    {
                        'time': '-'.join(l[:speaker_break].split()[:-1]),
                        'speaker': l[:speaker_break].split()[-1].replace(':', ''),
                        'text': l[speaker_break:].strip(),
                    }
                ]
    
    df = pd.DataFrame(df)
    df['corpus'] = 'callfriend-ko'
    df['conversation_id'] = conversation_id
    df['utterance_no'] = df.index
    
    df.to_csv(
        os.path.join(tsvs_path, new_file), 
        index=False,
        encoding='utf-8',
        sep='\t'
    )

  0%|          | 0/100 [00:00<?, ?it/s]

## Load in formatted data

In [7]:
all_files = grab_all_files(tsvs_path, '.tsv')
len(all_files), all_files[0]

(100, '../data/callfriend-ko/transcr/tsvs/ko_4012.tsv')

In [8]:
data = []
for f in all_files:
        
        data += [pd.read_table(f, sep='\t')]

data = pd.concat(data, ignore_index=True)

In [9]:
data.shape

(42750, 6)

In [10]:
data.head()

,time,speaker,text,corpus,conversation_id,utterance_no
0,67.35-71.29,B,<English recommend> =하는 게 있어. 그거는 <English req...,callfriend-ko,4012,0
1,68.79-71.89,A,%응 %응,callfriend-ko,4012,1
2,71.36-75.44,B,+그러니까 그거는 선생님이 해 주셔야 되고 그 다음에 우리 <English desi...,callfriend-ko,4012,2
3,75.23-77.98,A,%응 %응,callfriend-ko,4012,3
4,75.45-80.24,B,%아 <English office of school> =인가 거기에서 우리가 왜 그...,callfriend-ko,4012,4


In [12]:
re.sub(r'[a-z]+', '', data['text'].loc[0].lower())

'< > =하는 게 있어. 그거는 < > =가 아닌데'

In [13]:
data = data.rename(columns={'CHAT_ID': 'conversation_id', 'SPEAKER_ID': 'speaker', 'TEXT': 'text'})

In [14]:
data['conversation_id'].nunique()

100

In [15]:
data['conversation_id'] = data['corpus'] + '-' + data['conversation_id'].astype(str)

In [16]:
data.head()

,time,speaker,text,corpus,conversation_id,utterance_no
0,67.35-71.29,B,<English recommend> =하는 게 있어. 그거는 <English req...,callfriend-ko,callfriend-ko-4012,0
1,68.79-71.89,A,%응 %응,callfriend-ko,callfriend-ko-4012,1
2,71.36-75.44,B,+그러니까 그거는 선생님이 해 주셔야 되고 그 다음에 우리 <English desi...,callfriend-ko,callfriend-ko-4012,2
3,75.23-77.98,A,%응 %응,callfriend-ko,callfriend-ko-4012,3
4,75.45-80.24,B,%아 <English office of school> =인가 거기에서 우리가 왜 그...,callfriend-ko,callfriend-ko-4012,4


### Correcting utterances/removing CLAN specific formatting.

In [17]:
import re

In [18]:
def corrected_text(text, contraction_replacement_nonce='CCOONNTTRRAACCTTIIOONN', non_english=False):
    res = str(text)
    if non_english:
        res = re.sub(r'[a-z]+', '', res.lower())
    
    res = re.sub(r'\(\(.*?\)\)', ' ', res)
    # res = re.sub(r'\[.*?\]', ' ', res)
    
    # find contractions and preserve them . . .
    contractions = list(re.findall(r"\w+'\w+", res))
    for contraction in set(contractions):
        replacement = contraction.replace("'", contraction_replacement_nonce)
        res = res.replace(contraction, replacement)
    res = re.sub(r"(⌋|⌊|⌉|⌈)", '', res)
    
    res = res.replace(':', '')
    res = res.replace('\t', ' ')
    
    # remove numbers in parentheses (times???)
    res = re.sub(r'\(\d\.\d\)', ' ', res)
    
    # remove all other special characters.
    res = re.sub(r'[^\w\s]', ' ', res)
    
    
        
    res = re.sub(r'\s+', ' ', res).replace('[', ' ').replace(']', ' ').replace(contraction_replacement_nonce, "'")
    
    return res.strip()

In [19]:
data['raw_text'] = data['text'].values
data['text'] = [corrected_text(text,non_english=True) for text in tqdm(data['raw_text'].values)]

  0%|          | 0/42750 [00:00<?, ?it/s]

In [20]:
data[['corpus', 'raw_text', 'text']].head(n=6)

,corpus,raw_text,text
0,callfriend-ko,<English recommend> =하는 게 있어. 그거는 <English req...,하는 게 있어 그거는 가 아닌데
1,callfriend-ko,%응 %응,응 응
2,callfriend-ko,+그러니까 그거는 선생님이 해 주셔야 되고 그 다음에 우리 <English desi...,그러니까 그거는 선생님이 해 주셔야 되고 그 다음에 우리
3,callfriend-ko,%응 %응,응 응
4,callfriend-ko,%아 <English office of school> =인가 거기에서 우리가 왜 그...,아 인가 거기에서 우리가 왜 그 학생이었다는 거를
5,callfriend-ko,%응.,응


## Create juxtaposed corpus: (x,y) pairs

In [21]:
max_turns_apart = 200

In [22]:
import warnings; warnings.filterwarnings("ignore")

corpus = []
for cid in tqdm(data['conversation_id'].unique()):
    sub = data.loc[data['conversation_id'].isin([cid])]
    sub_index = sub.index.values
    
    for i in sub_index:
        if i != sub_index[-1]:
            
            # speaker vs. other
            next_line_no = ( (sub_index > i) & (~sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]
            
            # speaker vs. self 
            next_line_no = ( (sub_index > i) & (sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]

  0%|          | 0/100 [00:00<?, ?it/s]

In [23]:
data = pd.DataFrame(corpus)
print(data.shape)
data.head()

(9170611, 11)


,time,speaker,text,corpus,conversation_id,utterance_no,raw_text,next_speaker,next_text,next_utterance_no,next_utterance_delta_no
0,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,<English recommend> =하는 게 있어. 그거는 <English req...,A,응 응,1,0
1,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,<English recommend> =하는 게 있어. 그거는 <English req...,A,응 응,3,1
2,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,<English recommend> =하는 게 있어. 그거는 <English req...,A,응,5,2
3,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,<English recommend> =하는 게 있어. 그거는 <English req...,A,응 완전히 복잡하네,7,3
4,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,<English recommend> =하는 게 있어. 그거는 <English req...,A,가 누구야 언니,9,4


In [24]:
del data['raw_text']

In [25]:
rename_columns = dict()
for col in data.columns:
    if col == 'text':
        rename_columns[col] = 'x_utterance'
    elif col == 'next_text':
        rename_columns[col] = 'y_utterance'
    elif 'next_' in col:
        rename_columns[col]  = col.replace('next', 'y')
    else:
        rename_columns[col] = col

In [26]:
data = data.rename(columns=rename_columns)
data.head()

,time,speaker,x_utterance,corpus,conversation_id,utterance_no,y_speaker,y_utterance,y_utterance_no,y_utterance_delta_no
0,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,A,응 응,1,0
1,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,A,응 응,3,1
2,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,A,응,5,2
3,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,A,응 완전히 복잡하네,7,3
4,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,A,가 누구야 언니,9,4


In [27]:
data = data.rename(columns={'speaker':'x_speaker','utterance_no': 'x_turn_id', 'y_utterance_no': 'y_turn_id'})
data.head()

,time,x_speaker,x_utterance,corpus,conversation_id,x_turn_id,y_speaker,y_utterance,y_turn_id,y_utterance_delta_no
0,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,A,응 응,1,0
1,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,A,응 응,3,1
2,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,A,응,5,2
3,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,A,응 완전히 복잡하네,7,3
4,67.35-71.29,B,하는 게 있어 그거는 가 아닌데,callfriend-ko,callfriend-ko-4012,0,A,가 누구야 언니,9,4


In [28]:
data['x_speaker'] = data['conversation_id'].astype(str) + '-' + data['x_speaker'].astype(str)

data['y_speaker'] = data['conversation_id'].astype(str) + '-' + data['y_speaker'].astype(str)

In [29]:
data['self'] = data['x_speaker'] == data['y_speaker']
data = data.sort_values(by=['corpus', 'conversation_id', 'x_turn_id', 'self', 'y_turn_id'])
data.index = range(len(data))

In [32]:
data['self'].value_counts()/(data['self'].value_counts().sum())

self
False    0.504976
True     0.495024
Name: count, dtype: float64

In [33]:
data[['corpus', 'x_utterance', 'y_utterance']].isna().mean()

corpus         0.0
x_utterance    0.0
y_utterance    0.0
dtype: float64

In [30]:
# # corpus sanity check . . . make sure that conversation_ids are all unique 
# k = data[['conversation_id', 'corpus']].drop_duplicates()
# k['conversation_id'].nunique(), k['conversation_id'].nunique()/len(k)

In [34]:
data[['corpus', 'x_utterance', 'x_speaker', 'y_speaker', 'y_utterance', 'x_turn_id', 'y_turn_id']].head()

,corpus,x_utterance,x_speaker,y_speaker,y_utterance,x_turn_id,y_turn_id
0,callfriend-ko,하는 게 있어 그거는 가 아닌데,callfriend-ko-4012-B,callfriend-ko-4012-A,응 응,0,1
1,callfriend-ko,하는 게 있어 그거는 가 아닌데,callfriend-ko-4012-B,callfriend-ko-4012-A,응 응,0,3
2,callfriend-ko,하는 게 있어 그거는 가 아닌데,callfriend-ko-4012-B,callfriend-ko-4012-A,응,0,5
3,callfriend-ko,하는 게 있어 그거는 가 아닌데,callfriend-ko-4012-B,callfriend-ko-4012-A,응 완전히 복잡하네,0,7
4,callfriend-ko,하는 게 있어 그거는 가 아닌데,callfriend-ko-4012-B,callfriend-ko-4012-A,가 누구야 언니,0,9


In [35]:
data.shape

(9170611, 11)

## Save outputs for server operations.

In [36]:
# data.to_csv(outputs_path, index=False, encoding='utf-8', sep='\t')
data.to_parquet(outputs_path, engine='fastparquet', compression='snappy')